# Prática — Árvore Binária de Busca —  ABB (ou *Binary Search Tree — BST*)

**Estruturas de Dados** · Atividade assíncrona
**Liberação:** 24/09/2026 · **Entrega:** 01/10/2026

## Instruções

1. Preencha as células marcadas com `# SEU CÓDIGO AQUI` (apague o `raise NotImplementedError`).
2. Execute as células de **teste** logo abaixo de cada questão. Todas devem terminar com ✅.
3. Responda as perguntas discursivas nas células **✍️ Sua resposta**.
4. **Não use** bibliotecas prontas de árvore, nem `sorted`, `dict` ou `set` para *resolver* as questões (usar `collections.deque` como fila no percurso em largura é permitido).
5. **Antes de enviar:** use *Kernel → Restart & Run All* e confira se todas as saídas estão visíveis. Envie o arquivo `.ipynb`.

> Nome: Richard Matheus Bezerra Ataliba · Matrícula: ______________


---
## Questão 1 — Implementando a ABB

Implemente uma Árvore Binária de Busca que guarda pares **(chave, valor)**, com **chaves únicas**.

### Parte A — Operações básicas

| Método | Comportamento esperado |
|---|---|
| `adicionar(chave, valor=None)` | Insere o par. Se a chave **já existir**, apenas atualiza o valor (o tamanho não muda). **Iterativo.** |
| `buscar(chave)` | Retorna o valor da chave, ou `None` se ela não existir. Atualiza `self.comparacoes` com o **número de nós visitados**. **Iterativo.** |
| `contem(chave)` | Retorna `True`/`False`. |
| `remover(chave)` | Remove a chave e retorna `True`; retorna `False` se ela não existir. Para um nó com **dois filhos**, use o **sucessor** (menor chave da subárvore direita). |
| `minimo()` / `maximo()` | Retorna a menor/maior chave, ou `None` se a árvore estiver vazia. |
| `profundidade(chave)` | Nº de arestas da raiz até a chave (raiz = 0), ou `None` se ela não existir. **Iterativo.** |

### Parte B — Percursos

| Método | Ordem de visita |
|---|---|
| `em_ordem(processar)` | esquerda → **nó** → direita |
| `pre_ordem(processar)` | **nó** → esquerda → direita |
| `pos_ordem(processar)` | esquerda → direita → **nó** |
| `em_largura(processar)` | nível 0, nível 1, nível 2… (use uma fila) |

Os quatro percursos recebem uma função `processar` e a chamam **uma vez em cada nó**, passando o próprio nó:

```python
def processar(no):
    print(no.chave, no.valor)

arvore.em_ordem(processar)
```

Assim o mesmo percurso serve para imprimir, somar, coletar em uma lista ou qualquer outra tarefa: o que muda é só o `processar`.

> 💡 Por que `adicionar`, `buscar` e `profundidade` **precisam** ser iterativos? Um dos testes cria uma árvore com 3.000 nós em fila. Pense na pilha de chamadas.

In [1]:
class No:
    def __init__(self, chave, valor=None):
        self.chave = chave
        self.valor = valor
        self.esquerda = None
        self.direita = None

    def __repr__(self):
        return f"No({self.chave!r})"

In [2]:
from collections import deque


class ArvoreBinariaBusca:
    def __init__(self):
        self.raiz = None
        self.tamanho = 0
        self.comparacoes = 0   # nós visitados na última chamada a buscar()

    def __len__(self):
        return self.tamanho

    # ---------- Parte A: operações básicas ----------

    def adicionar(self, chave, valor=None):
        if self.raiz is None:
            self.raiz = No(chave, valor)
            self.tamanho = 1
            return
        atual = self.raiz
        while True:
            if chave == atual.chave:
                atual.valor = valor
                return
            if chave < atual.chave:
                if atual.esquerda is None:
                    atual.esquerda = No(chave, valor)
                    self.tamanho += 1
                    return
                atual = atual.esquerda
            else:
                if atual.direita is None:
                    atual.direita = No(chave, valor)
                    self.tamanho += 1
                    return
                atual = atual.direita

    def buscar(self, chave):
        self.comparacoes = 0
        atual = self.raiz
        while atual is not None:
            self.comparacoes += 1
            if chave == atual.chave:
                return atual.valor
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
        return None

    def contem(self, chave):
        atual = self.raiz
        while atual is not None:
            if chave == atual.chave:
                return True
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
        return False

    def remover(self, chave):
        pai = None
        atual = self.raiz
        eh_esquerda = False
        while atual is not None and atual.chave != chave:
            pai = atual
            if chave < atual.chave:
                atual = atual.esquerda
                eh_esquerda = True
            else:
                atual = atual.direita
                eh_esquerda = False
        if atual is None:
            return False

        # Caso 2 filhos: substitui pelo sucessor e remove o sucessor
        if atual.esquerda is not None and atual.direita is not None:
            suc_pai = atual
            sucessor = atual.direita
            while sucessor.esquerda is not None:
                suc_pai = sucessor
                sucessor = sucessor.esquerda
            atual.chave = sucessor.chave
            atual.valor = sucessor.valor
            # agora remove o sucessor (tem no máximo filho direito)
            pai = suc_pai
            atual = sucessor
            eh_esquerda = (pai.esquerda is atual)

        # Caso 0 ou 1 filho
        filho = atual.esquerda if atual.esquerda is not None else atual.direita
        if pai is None:
            self.raiz = filho
        elif eh_esquerda:
            pai.esquerda = filho
        else:
            pai.direita = filho
        self.tamanho -= 1
        return True

    def minimo(self):
        if self.raiz is None:
            return None
        atual = self.raiz
        while atual.esquerda is not None:
            atual = atual.esquerda
        return atual.chave

    def maximo(self):
        if self.raiz is None:
            return None
        atual = self.raiz
        while atual.direita is not None:
            atual = atual.direita
        return atual.chave

    def profundidade(self, chave):
        profund = 0
        atual = self.raiz
        while atual is not None:
            if chave == atual.chave:
                return profund
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
            profund += 1
        return None

    # ---------- Parte B: percursos ----------

    def em_ordem(self, processar):
        def _em_ordem(no):
            if no is None:
                return
            _em_ordem(no.esquerda)
            processar(no)
            _em_ordem(no.direita)
        _em_ordem(self.raiz)

    def pre_ordem(self, processar):
        def _pre_ordem(no):
            if no is None:
                return
            processar(no)
            _pre_ordem(no.esquerda)
            _pre_ordem(no.direita)
        _pre_ordem(self.raiz)

    def pos_ordem(self, processar):
        def _pos_ordem(no):
            if no is None:
                return
            _pos_ordem(no.esquerda)
            _pos_ordem(no.direita)
            processar(no)
        _pos_ordem(self.raiz)

    def em_largura(self, processar):
        if self.raiz is None:
            return
        fila = deque([self.raiz])
        while fila:
            no = fila.popleft()
            processar(no)
            if no.esquerda is not None:
                fila.append(no.esquerda)
            if no.direita is not None:
                fila.append(no.direita)


In [3]:
# ===== FERRAMENTA AUXILIAR (não altere) =====
# Transforma qualquer percurso em uma lista de chaves — útil nos testes e nas questões.
def coletar(percurso):
    """Ex.: coletar(arvore.em_ordem) -> [20, 30, 40, ...]"""
    chaves = []
    percurso(lambda no: chaves.append(no.chave))
    return chaves

In [4]:
# ===== TESTES DA QUESTÃO 1 — Parte A (não altere) =====
import random

def testar_q1a():
    a = ArvoreBinariaBusca()
    assert len(a) == 0 and a.raiz is None, "Árvore nova deve estar vazia"
    assert a.buscar(10) is None and a.minimo() is None and a.maximo() is None
    assert a.remover(10) is False

    for c in [50, 30, 70, 20, 40, 60, 80]:
        a.adicionar(c, f"v{c}")
    assert len(a) == 7, "tamanho incorreto após inserções"
    r = a.raiz
    assert (r.chave, r.esquerda.chave, r.direita.chave) == (50, 30, 70), "estrutura errada"
    assert (r.esquerda.esquerda.chave, r.esquerda.direita.chave) == (20, 40), "estrutura errada"
    assert (r.direita.esquerda.chave, r.direita.direita.chave) == (60, 80), "estrutura errada"

    assert a.buscar(60) == "v60" and a.comparacoes == 3, "buscar(60) deve visitar 3 nós"
    assert a.buscar(65) is None and a.comparacoes == 3, "buscar(65) deve visitar 3 nós e falhar"
    assert a.contem(20) and not a.contem(25)

    a.adicionar(60, "novo")
    assert len(a) == 7 and a.buscar(60) == "novo", "chave repetida deve só atualizar o valor"

    assert a.minimo() == 20 and a.maximo() == 80
    assert a.profundidade(50) == 0 and a.profundidade(40) == 2 and a.profundidade(99) is None

    # Remoção: folha, um filho, dois filhos
    assert a.remover(20) is True and not a.contem(20) and len(a) == 6, "falha ao remover folha"
    assert a.remover(30) is True and a.raiz.esquerda.chave == 40, "falha ao remover nó com um filho"
    assert a.remover(50) is True and a.raiz.chave == 60, "dois filhos: a raiz deveria virar o sucessor (60)"
    assert len(a) == 4 and all(a.contem(c) for c in [40, 60, 70, 80])
    assert a.remover(12345) is False and len(a) == 4
    for c in [40, 60, 70, 80]:
        assert a.remover(c)
    assert len(a) == 0 and a.raiz is None, "árvore deveria estar vazia"

    # Sucessor que tem filho à direita
    b = ArvoreBinariaBusca()
    for c in [50, 30, 70, 60, 80, 65]:
        b.adicionar(c)
    b.remover(50)
    assert b.raiz.chave == 60 and b.raiz.direita.esquerda.chave == 65, "o filho do sucessor se perdeu"

    # Sucessor que é o próprio filho direito
    b = ArvoreBinariaBusca()
    for c in [50, 30, 70, 80]:
        b.adicionar(c)
    b.remover(50)
    assert b.raiz.chave == 70 and b.raiz.esquerda.chave == 30 and b.raiz.direita.chave == 80

    # Teste de estresse contra uma referência
    rng = random.Random(7)
    ref = {}
    e = ArvoreBinariaBusca()
    for _ in range(3000):
        c = rng.randrange(300)
        if rng.random() < 0.6:
            e.adicionar(c, c * 2)
            ref[c] = c * 2
        else:
            assert e.remover(c) == (c in ref), f"remover({c}) retornou valor errado"
            ref.pop(c, None)
        assert len(e) == len(ref), "tamanho divergiu durante o teste de estresse"
    for c in range(300):
        assert e.contem(c) == (c in ref) and e.buscar(c) == ref.get(c)
    assert e.minimo() == min(ref) and e.maximo() == max(ref)

    # Árvore em fila com 3.000 nós (por isso os métodos acima são iterativos)
    d = ArvoreBinariaBusca()
    for c in range(3000):
        d.adicionar(c)
    assert d.profundidade(2999) == 2999
    d.buscar(2999)
    assert d.comparacoes == 3000

    print("✅ Questão 1 — Parte A: todos os testes passaram!")

testar_q1a()

✅ Questão 1 — Parte A: todos os testes passaram!


In [5]:
# ===== TESTES DA QUESTÃO 1 — Parte B (não altere) =====
def testar_q1b():
    vazia = ArvoreBinariaBusca()
    for percurso in (vazia.em_ordem, vazia.pre_ordem, vazia.pos_ordem, vazia.em_largura):
        assert coletar(percurso) == [], "percurso em árvore vazia deve não visitar nada"

    a = ArvoreBinariaBusca()
    for c in [50, 30, 70, 20, 40, 60, 80]:
        a.adicionar(c, f"v{c}")

    assert coletar(a.em_ordem) == [20, 30, 40, 50, 60, 70, 80], "em ordem deve sair em ordem crescente"
    assert coletar(a.pre_ordem) == [50, 30, 20, 40, 70, 60, 80], "pré-ordem incorreta"
    assert coletar(a.pos_ordem) == [20, 40, 30, 60, 80, 70, 50], "pós-ordem incorreta"
    assert coletar(a.em_largura) == [50, 30, 70, 20, 40, 60, 80], "em largura incorreta"

    # processar recebe o NÓ, não a chave
    pares = []
    a.em_ordem(lambda no: pares.append((no.chave, no.valor)))
    assert pares[0] == (20, "v20") and pares[-1] == (80, "v80"), "processar deve receber o nó"

    # processar é chamado exatamente uma vez por nó
    contagem = [0]
    a.pos_ordem(lambda no: contagem.__setitem__(0, contagem[0] + 1))
    assert contagem[0] == len(a)

    # árvore torta: um nó por nível
    t = ArvoreBinariaBusca()
    for c in [10, 20, 30, 40]:
        t.adicionar(c)
    assert coletar(t.em_largura) == [10, 20, 30, 40]
    assert coletar(t.pos_ordem) == [40, 30, 20, 10]

    # em largura visita nível a nível, e não ramo a ramo
    g = ArvoreBinariaBusca()
    for c in [50, 30, 70, 20, 80, 10]:
        g.adicionar(c)
    assert coletar(g.em_largura) == [50, 30, 70, 20, 80, 10]

    print("✅ Questão 1 — Parte B: todos os testes passaram!")

testar_q1b()

✅ Questão 1 — Parte B: todos os testes passaram!


---
## Questão 2 — Cadastro acadêmico: listagens e a forma da árvore

A secretaria guarda as matrículas em uma lista, e as consultas estão lentas. Você foi encarregado(a) de migrar o cadastro para a sua ABB.

As matrículas têm o formato **ano + sequencial** (ex.: `20240137`). O arquivo exportado pelo sistema antigo traz os alunos **na ordem em que foram cadastrados**.

**(a)** Crie `cadastro` (uma `ArvoreBinariaBusca`) e insira todos os alunos de `alunos`, **na ordem do arquivo**, com chave = matrícula e valor = nome.

**(b)** `consultar(arvore, matricula)`: retorna o nome, ou a string `"Matrícula não encontrada"`.

**(c)** `processar_trancamentos(arvore, lista)`: remove cada matrícula da lista e retorna `(removidas, nao_encontradas)`.

**(d)** `listar_por_matricula(arvore)`: retorna a lista de tuplas `(matrícula, nome)` **em ordem crescente de matrícula**, usando um percurso.

**(e)** `primeiros_visitados(arvore, percurso, k)`: retorna as `k` primeiras **chaves** visitadas pelo percurso recebido (por exemplo, `arvore.em_largura`). Dica: você pode parar de acumular depois de `k`, mas não precisa interromper o percurso.

In [6]:
# ===== DADOS (não altere) =====
import random

_rng = random.Random(2026)
_NOMES = ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Felipe", "Gabriela", "Heitor",
          "Isabela", "João", "Larissa", "Marcos", "Natália", "Otávio", "Paula", "Rafael"]
_SOBRENOMES = ["Silva", "Souza", "Oliveira", "Lima", "Costa", "Pereira", "Alves",
               "Ferreira", "Rodrigues", "Gomes", "Martins", "Araújo"]

def _gerar_alunos(ano, quantidade):
    return [(ano * 10000 + i, f"{_rng.choice(_NOMES)} {_rng.choice(_SOBRENOMES)}")
            for i in range(1, quantidade + 1)]

alunos = (_gerar_alunos(2023, 150) + _gerar_alunos(2024, 150)
          + _gerar_alunos(2025, 150) + _gerar_alunos(2026, 150))
trancamentos = _rng.sample([m for m, _ in alunos], 40) + [20270001, 20220150, 20259999]
_rng.shuffle(trancamentos)

print(len(alunos), "alunos | primeiros:", alunos[:3])

600 alunos | primeiros: [(20230001, 'Diego Pereira'), (20230002, 'Diego Lima'), (20230003, 'Otávio Gomes')]


In [7]:
# (a)
cadastro = ArvoreBinariaBusca()
for matricula, nome in alunos:
    cadastro.adicionar(matricula, nome)


# (b)
def consultar(arvore, matricula):
    if not arvore.contem(matricula):
        return "Matrícula não encontrada"
    return arvore.buscar(matricula)


# (c)
def processar_trancamentos(arvore, lista):
    removidas = 0
    nao_encontradas = 0
    for matricula in lista:
        if arvore.remover(matricula):
            removidas += 1
        else:
            nao_encontradas += 1
    return (removidas, nao_encontradas)


# (d)
def listar_por_matricula(arvore):
    resultado = []
    arvore.em_ordem(lambda no: resultado.append((no.chave, no.valor)))
    return resultado


# (e)
def primeiros_visitados(arvore, percurso, k):
    chaves = []
    def processar(no):
        if len(chaves) < k:
            chaves.append(no.chave)
    percurso(processar)
    return chaves


In [8]:
# ===== TESTES DA QUESTÃO 2 — parte 1 (não altere) =====
assert len(cadastro) == 600, "o cadastro deveria ter 600 alunos"
assert consultar(cadastro, alunos[10][0]) == alunos[10][1]
assert consultar(cadastro, 19990001) == "Matrícula não encontrada"

lista = listar_por_matricula(cadastro)
assert len(lista) == 600 and lista[0] == alunos[0] and lista[-1] == alunos[-1]
assert all(lista[i][0] < lista[i + 1][0] for i in range(len(lista) - 1)), "a listagem não saiu ordenada"
assert primeiros_visitados(cadastro, cadastro.em_largura, 3) == [a[0] for a in alunos[:3]]
print("✅ Questão 2 (a), (b), (d) e (e) ok!")

✅ Questão 2 (a), (b), (d) e (e) ok!


In [9]:
# Processando os trancamentos (execute uma única vez;
# se precisar repetir, execute novamente a célula do item (a))
resultado = processar_trancamentos(cadastro, trancamentos)
print("Removidas / não encontradas:", resultado)

Removidas / não encontradas: (40, 3)


In [10]:
# ===== TESTES DA QUESTÃO 2 — parte 2 (não altere) =====
assert resultado == (40, 3), "contagem de trancamentos incorreta"
assert len(cadastro) == 560
assert all(consultar(cadastro, m) == "Matrícula não encontrada" for m in trancamentos)
print("✅ Questão 2 (c) ok!")

✅ Questão 2 (c) ok!


In [11]:
# (f) Compare os quatro percursos no cadastro e imprima:
#     - as 5 primeiras chaves visitadas em ordem, em pré-ordem e em largura;
#     - se as três listas completas são iguais entre si;
#     - as 5 primeiras chaves da pós-ordem;
#     - a profundidade da maior matrícula.
primeiros_em_ordem = primeiros_visitados(cadastro, cadastro.em_ordem, 5)
primeiros_pre = primeiros_visitados(cadastro, cadastro.pre_ordem, 5)
primeiros_largura = primeiros_visitados(cadastro, cadastro.em_largura, 5)
print("5 primeiras (em ordem):", primeiros_em_ordem)
print("5 primeiras (pré-ordem):", primeiros_pre)
print("5 primeiras (em largura):", primeiros_largura)

lista_em_ordem = coletar(cadastro.em_ordem)
lista_pre = coletar(cadastro.pre_ordem)
lista_largura = coletar(cadastro.em_largura)
iguais = (lista_em_ordem == lista_pre == lista_largura)
print("As três listas completas são iguais?", iguais)

primeiros_pos = primeiros_visitados(cadastro, cadastro.pos_ordem, 5)
print("5 primeiras (pós-ordem):", primeiros_pos)

maior = cadastro.maximo()
prof_maior = cadastro.profundidade(maior)
print("Maior matrícula:", maior, "| profundidade:", prof_maior)


5 primeiras (em ordem): [20230001, 20230003, 20230004, 20230005, 20230006]
5 primeiras (pré-ordem): [20230001, 20230003, 20230004, 20230005, 20230006]
5 primeiras (em largura): [20230001, 20230003, 20230004, 20230005, 20230006]
As três listas completas são iguais? True
5 primeiras (pós-ordem): [20260150, 20260149, 20260148, 20260147, 20260146]
Maior matrícula: 20260150 | profundidade: 559


**(g)** Responda:

1. Em uma árvore qualquer, os percursos em ordem, pré-ordem e em largura dão resultados diferentes. Por que neste cadastro os três deram exatamente a mesma lista?
2. O que isso revela sobre o formato da árvore? Qual valor de profundidade você obteve?
3. Por que a pós-ordem saiu invertida?
4. Se a universidade tivesse 50.000 alunos, o que aconteceria ao chamar `em_ordem` nesta árvore? (Dica: `sys.getrecursionlimit()`.)

✍️ **Sua resposta:**

1. Neste cadastro as matrículas foram inseridas em ordem crescente (ano a ano, sequencialmente). Com isso, cada nova chave é maior que todas as anteriores e cai sempre à direita da raiz atual, formando uma “fila” (lista encadeada pela direita). Nessa árvore degenerada, o único caminho possível é descer sempre pela direita: em ordem (esquerda→nó→direita), pré-ordem (nó→esquerda→direita) e em largura (nível a nível) visitam exatamente a mesma sequência de nós.

2. Isso revela que a árvore está completamente desbalanceada — uma cadeia linear pela direita, com altura ≈ n−1. A profundidade da maior matrícula ficou em 559 (após os trancamentos restaram 560 alunos; a maior chave é a folha no fim da cadeia).

3. A pós-ordem visita esquerda→direita→nó. Como não há filhos à esquerda, ela desce até a folha mais à direita e só então sobe, imprimindo as chaves do maior para o menor — a sequência invertida da em-ordem.

4. Com 50.000 alunos nessa mesma forma de fila, a recursão de `em_ordem` precisaria de cerca de 50.000 frames na pilha. O limite padrão do Python (`sys.getrecursionlimit()`, tipicamente 1000) seria estourado e a chamada terminaria com `RecursionError`. Por isso métodos críticos em árvores potencialmente degeneradas devem ser iterativos.


---
## Questão 3 — Agenda de uma clínica: listagens e consultas por intervalo

Uma clínica atende das 08:00 às 17:40, com consultas a cada 20 minutos. Os pacientes ligam em qualquer ordem pedindo horários. A agenda do dia é uma ABB com **chave = horário** (string `"HH:MM"`) e **valor = nome do paciente**.

**(a)** `agendar(agenda, horario, paciente)`: agenda e retorna `True`. Se o horário **já estiver ocupado**, não altera nada e retorna `False`.

**(b)** `cancelar(agenda, horario)`: retorna `True` se havia consulta, `False` caso contrário.

**(c)** `listar_agenda(agenda)`: retorna a lista `(horário, paciente)` do primeiro ao último atendimento do dia.

**(d)** `listar_intervalo(agenda, inicio, fim)`: retorna a lista `(horário, paciente)` das consultas entre `inicio` e `fim` (inclusive), **com poda**: se a subárvore inteira estiver fora do intervalo, não desça nela. A função também deve contar quantos nós visitou e retornar `(consultas, visitados)`.

> 💡 A poda é o que diferencia esta consulta de "percorrer tudo e filtrar depois": em um nó com chave menor que `inicio`, toda a subárvore esquerda também é menor — não há motivo para visitá-la.

In [12]:
# ===== DADOS (não altere) =====
_rng = random.Random(15)
HORARIOS = [f"{h:02d}:{m:02d}" for h in range(8, 18) for m in (0, 20, 40)]
_PACIENTES = ["Aurora", "Benício", "Cecília", "Davi", "Eloá", "Gael", "Helena",
              "Ícaro", "Jade", "Levi", "Maitê", "Noah", "Olívia", "Pietro"]
pedidos = [(_rng.choice(HORARIOS), _rng.choice(_PACIENTES)) for _ in range(35)]
cancelamentos = ["08:00", "12:20", "17:40", "13:00"]
print(len(HORARIOS), "horários no dia |", len(pedidos), "pedidos | primeiros:", pedidos[:3])

30 horários no dia | 35 pedidos | primeiros: [('10:00', 'Aurora'), ('13:20', 'Noah'), ('08:20', 'Cecília')]


In [13]:
# (a)
def agendar(agenda, horario, paciente):
    if agenda.contem(horario):
        return False
    agenda.adicionar(horario, paciente)
    return True


# (b)
def cancelar(agenda, horario):
    return agenda.remover(horario)


# (c)
def listar_agenda(agenda):
    resultado = []
    agenda.em_ordem(lambda no: resultado.append((no.chave, no.valor)))
    return resultado


# (d)
def listar_intervalo(agenda, inicio, fim):
    consultas = []
    visitados = [0]

    def _intervalo(no):
        if no is None:
            return
        visitados[0] += 1
        if no.chave >= inicio:
            _intervalo(no.esquerda)
        if inicio <= no.chave <= fim:
            consultas.append((no.chave, no.valor))
        if no.chave <= fim:
            _intervalo(no.direita)

    _intervalo(agenda.raiz)
    return (consultas, visitados[0])


In [14]:
# ===== TESTES DA QUESTÃO 3 (não altere) =====
t = ArvoreBinariaBusca()
assert agendar(t, "10:00", "Ana") and agendar(t, "08:20", "Bia") and agendar(t, "14:40", "Caio")
assert agendar(t, "10:00", "Duda") is False and t.buscar("10:00") == "Ana", "não pode sobrescrever"
assert listar_agenda(t) == [("08:20", "Bia"), ("10:00", "Ana"), ("14:40", "Caio")]
assert cancelar(t, "10:00") and not cancelar(t, "10:00")
assert listar_agenda(t) == [("08:20", "Bia"), ("14:40", "Caio")]
assert listar_agenda(ArvoreBinariaBusca()) == []

# Intervalo com poda, em uma árvore de forma conhecida
p = ArvoreBinariaBusca()
for h in ["12:00", "10:00", "14:00", "09:00", "11:00", "13:00", "15:00", "08:00", "16:00"]:
    p.adicionar(h, "paciente " + h)
consultas, visitados = listar_intervalo(p, "10:30", "13:30")
assert [h for h, _ in consultas] == ["11:00", "12:00", "13:00"], "consultas do intervalo incorretas"
assert visitados <= 6, f"sem poda demais: {visitados} nós visitados de 9 (esperado no máximo 6)"
consultas, visitados = listar_intervalo(p, "08:00", "16:00")
assert len(consultas) == 9 and visitados == 9, "com o intervalo inteiro, todos os nós são visitados"
assert listar_intervalo(p, "17:00", "18:00")[0] == []

agenda = ArvoreBinariaBusca()
aceitos = sum(agendar(agenda, h, p_) for h, p_ in pedidos)
assert aceitos == len({h for h, _ in pedidos}) == len(agenda)
print("✅ Questão 3: testes passaram!")

✅ Questão 3: testes passaram!


In [15]:
# (e) Simule o dia e imprima:
agenda_dia = ArvoreBinariaBusca()
aceitos = 0
recusados = 0
for horario, paciente in pedidos:
    if agendar(agenda_dia, horario, paciente):
        aceitos += 1
    else:
        recusados += 1
print(f"Pedidos aceitos: {aceitos} | recusados: {recusados}")

print("\nAgenda completa:")
for horario, paciente in listar_agenda(agenda_dia):
    print(f"  {horario} — {paciente}")

print("\nCancelamentos:")
for horario in cancelamentos:
    ok = cancelar(agenda_dia, horario)
    print(f"  cancelar({horario!r}) -> {ok}")

manha, visitados_manha = listar_intervalo(agenda_dia, "08:00", "11:59")
print(f"\nTurno da manhã (08:00–11:59): {manha}")
print(f"Nós visitados na consulta por intervalo: {visitados_manha}")


Pedidos aceitos: 21 | recusados: 14

Agenda completa:
  08:00 — Aurora
  08:20 — Cecília
  08:40 — Ícaro
  09:00 — Gael
  10:00 — Aurora
  10:20 — Levi
  11:00 — Noah
  11:20 — Davi
  11:40 — Davi
  12:00 — Eloá
  12:20 — Ícaro
  12:40 — Noah
  13:20 — Noah
  13:40 — Cecília
  14:00 — Olívia
  14:40 — Ícaro
  15:00 — Cecília
  16:40 — Aurora
  17:00 — Noah
  17:20 — Olívia
  17:40 — Davi

Cancelamentos:
  cancelar('08:00') -> True
  cancelar('12:20') -> True
  cancelar('17:40') -> True
  cancelar('13:00') -> False

Turno da manhã (08:00–11:59): [('08:20', 'Cecília'), ('08:40', 'Ícaro'), ('09:00', 'Gael'), ('10:00', 'Aurora'), ('10:20', 'Levi'), ('11:00', 'Noah'), ('11:20', 'Davi'), ('11:40', 'Davi')]
Nós visitados na consulta por intervalo: 11


**(f)** Responda:

1. Compare `listar_intervalo` com a alternativa "chamar `listar_agenda` e filtrar depois". Qual o custo de cada uma em função de `n` (total de consultas) e de `m` (consultas dentro do intervalo)?
2. Em que situação a poda **não** economiza nada?
3. Os horários estão no formato `"HH:MM"` com dois dígitos. O que daria errado se a recepção cadastrasse `"9:40"` em vez de `"09:40"`? (Dica: compare `"9:40" < "10:00"` no Python.)

✍️ **Sua resposta:**

1. Filtrar depois de `listar_agenda` percorre os **n** nós da árvore (custo Θ(n)) e só então descarta os que estão fora do intervalo. Já `listar_intervalo` com poda visita apenas os nós no caminho até o intervalo e os **m** nós dentro dele; em uma ABB equilibrada o custo fica próximo de O(m + h), com h = O(log n), bem menor que n quando o intervalo é estreito.

2. A poda não economiza nada quando o intervalo cobre (quase) toda a árvore — por exemplo, de `"08:00"` a `"16:00"` no teste — ou quando a árvore está degenerada em fila e o intervalo inclui o fim da cadeia: quase todo nó acaba sendo visitado de qualquer forma.

3. A comparação lexicográfica de strings depende dos caracteres. `"9:40" < "10:00"` é **False** em Python, porque `'9' > '1'`. O horário das 9:40 ficaria “depois” das 10:00 na ABB, quebrando a ordem cronológica da agenda. Por isso o formato com dois dígitos (`"09:40"`) é essencial.


---
## Questão 4 — Catálogo de uma biblioteca: a forma importa

O sistema da biblioteca exporta o catálogo **ordenado por código**. Você vai carregar os mesmos 500 livros de duas maneiras e medir a diferença.

**(a)** `altura(arvore)`: retorna a altura da árvore em arestas (folha = 0, árvore vazia = −1). A altura de um nó depende da altura dos filhos, então ele só pode ser calculado **depois** deles — é o padrão da pós-ordem. Implemente com uma recursão nesse formato (uma função auxiliar `_altura(no)` ajuda).

**(b)** `contar_folhas(arvore)`: retorna quantos nós não têm filhos. Use qualquer um dos percursos da Questão 1 com uma função `processar`.

**(c)** `niveis(arvore)`: retorna uma **lista de listas** com as chaves de cada nível (`niveis[0]` é a raiz, `niveis[1]` são os filhos da raiz, e assim por diante). Use o percurso em largura com uma fila, processando um nível inteiro a cada volta do laço.

**(d)** Carregue `catalogo_planilha` na ordem do arquivo e `catalogo_balanceado` inserindo sempre o livro do **meio** primeiro (depois o meio de cada metade, recursivamente). Implemente `carregar_balanceado(arvore, livros_ordenados)`.

**(e)** Compare as duas árvores: altura, número de folhas, tamanho dos 4 primeiros níveis e média de nós visitados ao buscar 500 códigos sorteados.

**(f)** Descarte os livros de `descartados` (use `remover`) das duas árvores e mostre como altura e folhas mudaram.

In [16]:
# ===== DADOS (não altere) =====
_rng = random.Random(9)
_TITULOS = ["Algoritmos", "Cálculo", "Redes", "Compiladores", "Banco de Dados", "Sistemas Operacionais",
            "Estatística", "Física", "Inteligência Artificial", "Engenharia de Software",
            "Grafos", "Criptografia", "Robótica", "Geometria", "Lógica"]
_EDICOES = ["1ª ed.", "2ª ed.", "3ª ed.", "4ª ed.", "edição revisada", "edição comentada"]

livros = []           # (codigo, titulo) — já vem ordenado por código
_codigo = 100000
for _ in range(500):
    _codigo += _rng.randint(1, 40)
    livros.append((_codigo, f"{_rng.choice(_TITULOS)} — {_rng.choice(_EDICOES)}"))

descartados = _rng.sample([c for c, _ in livros], 60)
consultas_codigo = [_rng.choice(livros)[0] for _ in range(500)]
print(len(livros), "livros | primeiro:", livros[0], "| último:", livros[-1])

500 livros | primeiro: (100030, 'Engenharia de Software — 3ª ed.') | último: (109697, 'Física — 4ª ed.')


In [17]:
# (a)
def altura(arvore):
    def _altura(no):
        if no is None:
            return -1
        return 1 + max(_altura(no.esquerda), _altura(no.direita))
    return _altura(arvore.raiz)


# (b)
def contar_folhas(arvore):
    contagem = [0]
    def processar(no):
        if no.esquerda is None and no.direita is None:
            contagem[0] += 1
    arvore.em_ordem(processar)
    return contagem[0]


# (c)
def niveis(arvore):
    if arvore.raiz is None:
        return []
    resultado = []
    fila = deque([arvore.raiz])
    while fila:
        tamanho_nivel = len(fila)
        nivel = []
        for _ in range(tamanho_nivel):
            no = fila.popleft()
            nivel.append(no.chave)
            if no.esquerda is not None:
                fila.append(no.esquerda)
            if no.direita is not None:
                fila.append(no.direita)
        resultado.append(nivel)
    return resultado


# (d)
def carregar_balanceado(arvore, livros_ordenados):
    def _inserir_meio(inicio, fim):
        if inicio > fim:
            return
        meio = (inicio + fim) // 2
        codigo, titulo = livros_ordenados[meio]
        arvore.adicionar(codigo, titulo)
        _inserir_meio(inicio, meio - 1)
        _inserir_meio(meio + 1, fim)
    _inserir_meio(0, len(livros_ordenados) - 1)


catalogo_planilha = ArvoreBinariaBusca()
for codigo, titulo in livros:
    catalogo_planilha.adicionar(codigo, titulo)

catalogo_balanceado = ArvoreBinariaBusca()
carregar_balanceado(catalogo_balanceado, livros)


In [18]:
# ===== TESTES DA QUESTÃO 4 (não altere) =====
assert altura(ArvoreBinariaBusca()) == -1
_u = ArvoreBinariaBusca(); _u.adicionar(1)
assert altura(_u) == 0 and contar_folhas(_u) == 1 and niveis(_u) == [[1]]
assert niveis(ArvoreBinariaBusca()) == []

_t = ArvoreBinariaBusca()
for c in [50, 30, 70, 20, 40, 60, 80, 65]:
    _t.adicionar(c)
assert altura(_t) == 3 and contar_folhas(_t) == 4
assert niveis(_t) == [[50], [30, 70], [20, 40, 60, 80], [65]]

assert len(catalogo_planilha) == 500 and len(catalogo_balanceado) == 500
assert altura(catalogo_planilha) == 499, "carregado na ordem do arquivo, o catálogo vira uma fila"
assert altura(catalogo_balanceado) == 8, "o carregamento pelo meio deveria produzir altura 8"
assert contar_folhas(catalogo_planilha) == 1
assert [len(n) for n in niveis(catalogo_balanceado)][:4] == [1, 2, 4, 8]
assert coletar(catalogo_balanceado.em_ordem) == [c for c, _ in livros], "as chaves não conferem"
print("✅ Questão 4 (a) a (d) ok!")

✅ Questão 4 (a) a (d) ok!


In [19]:
# (e) Comparação
print("=== Comparação dos catálogos ===")
print(f"altura planilha:    {altura(catalogo_planilha)}")
print(f"altura balanceado:  {altura(catalogo_balanceado)}")
print(f"folhas planilha:    {contar_folhas(catalogo_planilha)}")
print(f"folhas balanceado:  {contar_folhas(catalogo_balanceado)}")

niveis_p = niveis(catalogo_planilha)
niveis_b = niveis(catalogo_balanceado)
print(f"tamanhos dos 4 primeiros níveis (planilha):   {[len(n) for n in niveis_p[:4]]}")
print(f"tamanhos dos 4 primeiros níveis (balanceado): {[len(n) for n in niveis_b[:4]]}")

def media_comparacoes(arvore, consultas):
    total = 0
    for codigo in consultas:
        arvore.buscar(codigo)
        total += arvore.comparacoes
    return total / len(consultas)

media_p = media_comparacoes(catalogo_planilha, consultas_codigo)
media_b = media_comparacoes(catalogo_balanceado, consultas_codigo)
print(f"média de nós visitados (planilha):   {media_p:.2f}")
print(f"média de nós visitados (balanceado): {media_b:.2f}")


=== Comparação dos catálogos ===
altura planilha:    499
altura balanceado:  8
folhas planilha:    1
folhas balanceado:  245
tamanhos dos 4 primeiros níveis (planilha):   [1, 1, 1, 1]
tamanhos dos 4 primeiros níveis (balanceado): [1, 2, 4, 8]
média de nós visitados (planilha):   261.94
média de nós visitados (balanceado): 8.01


In [20]:
# (f) Descarte dos livros e novo comparativo
for codigo in descartados:
    catalogo_planilha.remover(codigo)
    catalogo_balanceado.remover(codigo)

print("=== Após descartes ===")
print(f"tamanho planilha:   {len(catalogo_planilha)} | balanceado: {len(catalogo_balanceado)}")
print(f"altura planilha:    {altura(catalogo_planilha)} | balanceado: {altura(catalogo_balanceado)}")
print(f"folhas planilha:    {contar_folhas(catalogo_planilha)} | balanceado: {contar_folhas(catalogo_balanceado)}")


=== Após descartes ===
tamanho planilha:   440 | balanceado: 440
altura planilha:    439 | balanceado: 8
folhas planilha:    1 | balanceado: 199


**(g)** Responda:

1. Por que a altura precisa da pós-ordem (ou de uma recursão equivalente), e não pode ser calculada em pré-ordem?
2. Olhando os tamanhos dos níveis das duas árvores, o que cada formato revela?
3. Compare a média de nós visitados nas buscas com a altura de cada árvore. A relação bate com o esperado?
4. Depois dos descartes, a árvore balanceada continuou baixa? Isso prova que ela vai continuar assim para sempre? O que faltaria para garantir isso?

✍️ **Sua resposta:**

1. A altura de um nó é `1 + max(altura(esquerda), altura(direita))`. Esse valor só existe **depois** de conhecer a altura dos filhos; por isso o cálculo segue o padrão da pós-ordem. Em pré-ordem o nó seria visitado antes dos filhos, quando essas alturas ainda não estão disponíveis.

2. Na planilha, os 4 primeiros níveis têm tamanho `[1, 1, 1, 1]` — um nó por nível, típico de fila. No balanceado, `[1, 2, 4, 8]`: os níveis dobram, revelando uma árvore cheia/quase completa.

3. Sim. Na planilha a média de nós visitados foi ≈ 262, coerente com altura 499 (buscas percorrem em média ~n/2 nós). No balanceado a média foi ≈ 8,01, batendo com a altura 8 (≈ log₂ 500). A forma da árvore domina o custo da busca.

4. Depois dos 60 descartes a árvore balanceada manteve altura 8 (agora com 440 livros e 199 folhas). Isso **não** prova que ela permanecerá baixa para sempre: inserções futuras em ordem podem degenerá-la de novo. Para garantir balanceamento contínuo faltaria rebalanceamento automático (AVL, rubro-negra, etc.).


---
## Questão 5 — Backup e restauração do catálogo

A biblioteca quer **salvar o catálogo em um arquivo texto** no fim do dia e **restaurá-lo** na manhã seguinte. Qual percurso usar na exportação muda tudo.

**(a)** `exportar(arvore, percurso)`: retorna a lista de tuplas `(chave, valor)` na ordem do percurso recebido (ex.: `arvore.pre_ordem`).

**(b)** `restaurar(registros)`: cria e devolve uma nova `ArvoreBinariaBusca` inserindo os registros **na ordem da lista**.

**(c)** Restaure `catalogo_balanceado` a partir de duas exportações — uma em **pré-ordem** e outra **em ordem** — e compare as três árvores (original e as duas restauradas): altura, número de nós e a lista em ordem.

**(d)** Meça a média de nós visitados nas buscas de `consultas_codigo` nas duas árvores restauradas.

In [21]:
# (a)
def exportar(arvore, percurso):
    registros = []
    percurso(lambda no: registros.append((no.chave, no.valor)))
    return registros


# (b)
def restaurar(registros):
    arvore = ArvoreBinariaBusca()
    for chave, valor in registros:
        arvore.adicionar(chave, valor)
    return arvore


In [22]:
# ===== TESTES DA QUESTÃO 5 (não altere) =====
_t = ArvoreBinariaBusca()
for c in [50, 30, 70, 20, 40]:
    _t.adicionar(c, f"v{c}")
assert exportar(_t, _t.pre_ordem) == [(50, "v50"), (30, "v30"), (20, "v20"), (40, "v40"), (70, "v70")]
assert exportar(_t, _t.em_ordem)[0] == (20, "v20")

_copia = restaurar(exportar(_t, _t.pre_ordem))
assert len(_copia) == len(_t) and altura(_copia) == altura(_t)
assert coletar(_copia.pre_ordem) == coletar(_t.pre_ordem), "a pré-ordem deveria recriar a MESMA árvore"
assert _copia.buscar(40) == "v40"

_torta = restaurar(exportar(_t, _t.em_ordem))
assert coletar(_torta.em_ordem) == coletar(_t.em_ordem), "as chaves devem ser as mesmas"
assert altura(_torta) == len(_t) - 1, "restaurar a partir da em ordem gera uma fila"
print("✅ Questão 5 (a) e (b) ok!")

✅ Questão 5 (a) e (b) ok!


In [23]:
# (c) e (d)
export_pre = exportar(catalogo_balanceado, catalogo_balanceado.pre_ordem)
export_em = exportar(catalogo_balanceado, catalogo_balanceado.em_ordem)

restaurada_pre = restaurar(export_pre)
restaurada_em = restaurar(export_em)

print("=== Comparação das três árvores ===")
for nome, arv in [("original (balanceado)", catalogo_balanceado),
                  ("restaurada (pré-ordem)", restaurada_pre),
                  ("restaurada (em ordem)", restaurada_em)]:
    print(f"{nome}: n={len(arv)}, altura={altura(arv)}")

mesmas_chaves = (coletar(catalogo_balanceado.em_ordem)
                 == coletar(restaurada_pre.em_ordem)
                 == coletar(restaurada_em.em_ordem))
print("Listas em ordem idênticas?", mesmas_chaves)
print("Pré-ordem idêntica à original?",
      coletar(restaurada_pre.pre_ordem) == coletar(catalogo_balanceado.pre_ordem))

media_pre = media_comparacoes(restaurada_pre, consultas_codigo)
media_em = media_comparacoes(restaurada_em, consultas_codigo)
print(f"média de nós visitados — restaurada pré-ordem: {media_pre:.2f}")
print(f"média de nós visitados — restaurada em ordem:  {media_em:.2f}")


=== Comparação das três árvores ===
original (balanceado): n=440, altura=8
restaurada (pré-ordem): n=440, altura=8
restaurada (em ordem): n=440, altura=439
Listas em ordem idênticas? True
Pré-ordem idêntica à original? True
média de nós visitados — restaurada pré-ordem: 7.95
média de nós visitados — restaurada em ordem:  229.66


**(e)** Escreva um **parecer de 6 a 10 linhas** para a equipe da biblioteca, usando os números que **você** obteve:

1. As três árvores guardam as mesmas chaves. Então por que o backup em ordem é uma má escolha?
2. Por que a pré-ordem recria exatamente a mesma árvore? (Pense em qual nó ela grava primeiro.)
3. A pós-ordem também serviria para restaurar a árvore original? E o percurso em largura?
4. Qual formato de backup você recomenda, e o que aconteceria se a biblioteca dobrasse o acervo?

✍️ **Sua resposta:**

**Parecer técnico — backup do catálogo**

As três árvores guardam as mesmas 440 chaves (listagens em ordem idênticas), mas a forma depende do percurso do backup. Restaurar a partir da **em ordem** reinsere as chaves já ordenadas e reconstrói uma fila: altura 439 e média ≈ 230 nós visitados por busca — praticamente inutilizável. Já a exportação em **pré-ordem** grava a raiz primeiro e depois cada subárvore na ordem em que a inserção as recria; a árvore restaurada ficou idêntica à original (altura 8, mesma pré-ordem, média ≈ 8 comparações).

A **pós-ordem** não serve para recriar a forma: ela grava a raiz por último, e reinseri-la no fim não reconstrói as subárvores. O percurso **em largura** também preservaria a forma (raiz primeiro, depois os filhos nível a nível), mas a pré-ordem é o clássico e mais simples.

**Recomendação:** exportar sempre em **pré-ordem** (ou em largura). Se o acervo dobrar, a árvore restaurada pela pré-ordem continua com altura ≈ log₂(n); o backup em ordem viraria uma fila com o dobro do comprimento e tornaria as consultas inviáveis.


---
### ✅ Antes de enviar

- [x] *Kernel → Restart & Run All* executou sem erros
- [x] Todas as células de teste mostram ✅
- [x] Todas as respostas ✍️ foram preenchidas
- [x] Nome e matrícula no topo do notebook
